In [0]:
%run ./00_config

In [0]:
# ============================================================
# GOLD PRODUCT PERFORMANCE
# Source : billing.sales_data.billing_silver
# Target : billing.sales_data.gold_product_performance
#
# Grain:
#   orderDate + productName + source_date
#
# VM logic preserved:
#   - NO orderStatus filter
#   - Exclude Ho Marlboro
#   - Sales = SUM(totalProductPrice)
#   - Quantity Sold = SUM(quantity)
#   - Orders = COUNT(DISTINCT invoice)
#
# Existing target schema:
#   productName    STRING
#   orderDate      DATE
#   nooforders     BIGINT
#   sales          DOUBLE
#   quantitysold   BIGINT
#   source_date    STRING
#
# Write strategy:
#   - Atomic Delta replaceWhere
#   - No mergeSchema
#   - No schema evolution
#   - Exact target schema enforced before write
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    DateType,
    LongType,
    DoubleType
)

import uuid
from datetime import datetime, timezone


# ============================================================
# CONFIG
# ============================================================

SOURCE_TABLE = "billing.sales_data.billing_silver"
TARGET_TABLE = "billing.sales_data.gold_product_performance"

TARGET_DATE = "2026-10-03"

PIPELINE_RUN_ID = str(uuid.uuid4())

EXCLUDED_STORES = [
    "ho marlboro"
]


# ============================================================
# HELPERS
# ============================================================

def log(message):
    print(f"[GOLD_PRODUCT] {message}")


def fail(message):
    raise RuntimeError(f"[GOLD_PRODUCT] {message}")


def table_exists(table_name):
    return spark.catalog.tableExists(table_name)


# ============================================================
# START
# ============================================================

START_TIME = datetime.now(timezone.utc)

log("=" * 70)
log("PRODUCTION GOLD PRODUCT PERFORMANCE")
log(f"Pipeline run ID : {PIPELINE_RUN_ID}")
log(f"Target date     : {TARGET_DATE}")
log(f"Source table    : {SOURCE_TABLE}")
log(f"Target table    : {TARGET_TABLE}")
log("=" * 70)


# ============================================================
# SOURCE VALIDATION
# ============================================================

if not table_exists(SOURCE_TABLE):
    fail(f"Source table does not exist: {SOURCE_TABLE}")


silver_df = spark.table(SOURCE_TABLE)


required_columns = [
    "source_date",
    "orderDate",
    "productName",
    "storeName",
    "invoice",
    "totalProductPrice",
    "quantity"
]


missing_columns = [
    c for c in required_columns
    if c not in silver_df.columns
]


if missing_columns:
    fail(
        f"Missing required columns in Silver: "
        f"{missing_columns}"
    )


# ============================================================
# TARGET-DATE INPUT
# ============================================================

silver_target = (
    silver_df
    .filter(
        F.col("source_date").cast("string") == TARGET_DATE
    )
)


input_count = silver_target.count()

log(f"Silver rows: {input_count}")


if input_count == 0:
    fail(
        f"No Silver data found for target date "
        f"{TARGET_DATE}"
    )


# ============================================================
# STORE EXCLUSION
# ============================================================

filtered = (
    silver_target
    .filter(
        ~F.lower(
            F.trim(
                F.col("storeName")
            )
        ).isin(EXCLUDED_STORES)
    )
)


filtered_count = filtered.count()

log(f"Rows after exclusions: {filtered_count}")


if filtered_count == 0:
    fail(
        "No data remains after store exclusion"
    )


# ============================================================
# PREPARE
# ============================================================

prepared = (
    filtered

    # Business transaction date
    .withColumn(
        "_order_date",
        F.to_date(F.col("orderDate"))
    )

    # Sales must match target DOUBLE
    .withColumn(
        "_sales",
        F.col("totalProductPrice").cast("double")
    )

    # Quantity must ultimately become BIGINT
    # Keep source quantity numeric before aggregation.
    .withColumn(
        "_quantity",
        F.col("quantity").cast("double")
    )

    # Invoice used for distinct order count
    .withColumn(
        "_invoice",
        F.trim(
            F.col("invoice").cast("string")
        )
    )

    # Product name normalized consistently
    .withColumn(
        "_product_name",
        F.col("productName").cast("string")
    )
)


# ============================================================
# AGGREGATE
# ============================================================

gold_df = (
    prepared

    .groupBy(
        "_order_date",
        "_product_name"
    )

    .agg(

        # Existing target = BIGINT
        F.countDistinct("_invoice")
        .cast("long")
        .alias("nooforders"),

        # Existing target = DOUBLE
        F.sum("_sales")
        .cast("double")
        .alias("sales"),

        # Existing target = BIGINT
        F.sum("_quantity")
        .cast("long")
        .alias("quantitysold")
    )

    .withColumnRenamed(
        "_order_date",
        "orderDate"
    )

    .withColumnRenamed(
        "_product_name",
        "productName"
    )
)


# ============================================================
# SNAPSHOT COLUMN
# ============================================================

# IMPORTANT:
# Existing production table does NOT contain:
#   pipeline_run_id
#   processed_at
#
# Therefore they must NOT be added to gold_df.
#
# source_date is STRING in the existing target.

gold_df = (
    gold_df
    .withColumn(
        "source_date",
        F.lit(TARGET_DATE).cast("string")
    )
)


# ============================================================
# EXACT TARGET COLUMN ORDER + TYPES
# ============================================================

gold_df = (
    gold_df
    .select(
        F.col("productName").cast("string").alias("productName"),
        F.col("orderDate").cast("date").alias("orderDate"),
        F.col("nooforders").cast("long").alias("nooforders"),
        F.col("sales").cast("double").alias("sales"),
        F.col("quantitysold").cast("long").alias("quantitysold"),
        F.col("source_date").cast("string").alias("source_date")
    )
)


# ============================================================
# TARGET SCHEMA VALIDATION
# ============================================================

EXPECTED_COLUMNS = [
    "productName",
    "orderDate",
    "nooforders",
    "sales",
    "quantitysold",
    "source_date"
]


EXPECTED_TYPES = {
    "productName": "string",
    "orderDate": "date",
    "nooforders": "bigint",
    "sales": "double",
    "quantitysold": "bigint",
    "source_date": "string"
}


generated_schema = {
    field.name: field.dataType.simpleString()
    for field in gold_df.schema.fields
}


# ------------------------------------------------------------
# Column validation
# ------------------------------------------------------------

if gold_df.columns != EXPECTED_COLUMNS:
    fail(
        "Generated Gold columns do not match target schema.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {gold_df.columns}"
    )


# ------------------------------------------------------------
# Type validation
# ------------------------------------------------------------

for column_name, expected_type in EXPECTED_TYPES.items():

    actual_type = generated_schema[column_name]

    if actual_type != expected_type:

        fail(
            f"Schema mismatch for {column_name}: "
            f"expected={expected_type}, "
            f"actual={actual_type}"
        )


log("Generated Gold schema validation passed")


# ============================================================
# GOLD ROW COUNT
# ============================================================

gold_count = gold_df.count()

log(f"Gold rows: {gold_count}")


if gold_count == 0:
    fail("Gold output is empty")


# ============================================================
# DUPLICATE GRAIN VALIDATION
# ============================================================

duplicate_groups = (
    gold_df

    .groupBy(
        "orderDate",
        "productName",
        "source_date"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)


if duplicate_groups > 0:

    fail(
        f"Duplicate product grain detected: "
        f"{duplicate_groups} groups"
    )


log("Duplicate grain validation passed")


# ============================================================
# EXCLUDED STORE VALIDATION
# ============================================================
#
# StoreName is not part of Gold Product Performance grain.
# Therefore exclusion is validated against the input Silver
# dataframe rather than the Gold dataframe.
# ============================================================

excluded_store_rows = (
    silver_target
    .filter(
        F.lower(
            F.trim(
                F.col("storeName")
            )
        ).isin(EXCLUDED_STORES)
    )
    .count()
)


# Since this validation is only checking that the Gold source
# filtering logic was applied, we log the excluded input rows.

log(
    f"Rows belonging to excluded stores in Silver: "
    f"{excluded_store_rows}"
)


# ============================================================
# SILVER -> GOLD SALES RECONCILIATION
# ============================================================

silver_sales_row = (
    filtered

    .agg(
        F.sum(
            F.col("totalProductPrice")
            .cast("double")
        ).alias("sales")
    )

    .first()
)


silver_sales = float(
    silver_sales_row["sales"] or 0.0
)


gold_sales_row = (
    gold_df

    .agg(
        F.sum("sales").alias("sales")
    )

    .first()
)


gold_sales = float(
    gold_sales_row["sales"] or 0.0
)


difference = abs(
    silver_sales - gold_sales
)


log(
    f"Silver sales: {silver_sales:.4f}"
)

log(
    f"Gold sales  : {gold_sales:.4f}"
)

log(
    f"Difference  : {difference:.4f}"
)


if difference > 0.01:

    fail(
        "Sales reconciliation failed. "
        f"Difference={difference:.4f}"
    )


log(
    "Silver -> Gold sales reconciliation passed"
)


# ============================================================
# TARGET TABLE VALIDATION / CREATION
# ============================================================

if not table_exists(TARGET_TABLE):

    # --------------------------------------------------------
    # CREATE TARGET
    # --------------------------------------------------------

    log(
        "Target table does not exist. "
        "Creating production Gold table."
    )

    (
        gold_df

        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("source_date")
        .saveAsTable(TARGET_TABLE)
    )

    log(
        "Target table created successfully."
    )


else:

    # --------------------------------------------------------
    # EXISTING TARGET
    # --------------------------------------------------------

    target_df = spark.table(TARGET_TABLE)

    target_schema = target_df.schema


    log(
        f"Existing target columns: "
        f"{target_df.columns}"
    )


    # --------------------------------------------------------
    # EXACT TARGET COLUMN VALIDATION
    # --------------------------------------------------------

    if target_df.columns != EXPECTED_COLUMNS:

        fail(
            "Existing target columns do not match "
            "the expected production schema.\n"
            f"Expected: {EXPECTED_COLUMNS}\n"
            f"Actual:   {target_df.columns}"
        )


    # --------------------------------------------------------
    # EXACT TARGET TYPE VALIDATION
    # --------------------------------------------------------

    target_schema_dict = {
        field.name: field.dataType.simpleString()
        for field in target_schema.fields
    }


    for column_name, expected_type in EXPECTED_TYPES.items():

        actual_type = target_schema_dict[column_name]

        if actual_type != expected_type:

            fail(
                f"Existing target schema mismatch for "
                f"{column_name}: "
                f"expected={expected_type}, "
                f"actual={actual_type}"
            )


    log(
        "Existing target schema validation passed"
    )


    # --------------------------------------------------------
    # SOURCE DATE TYPE
    # --------------------------------------------------------

    source_date_field = next(
        (
            field
            for field in target_schema
            if field.name == "source_date"
        ),
        None
    )


    if source_date_field is None:

        fail(
            "Target table does not contain source_date"
        )


    log(
        f"Existing source_date type: "
        f"{source_date_field.dataType.simpleString()}"
    )


    if not isinstance(
        source_date_field.dataType,
        StringType
    ):

        fail(
            "Production target source_date is expected "
            "to be STRING."
        )


    # --------------------------------------------------------
    # ATOMIC REPLACEMENT
    # --------------------------------------------------------

    replace_predicate = (
        f"source_date = '{TARGET_DATE}'"
    )


    log(
        "Atomically replacing Gold snapshot:"
    )

    log(
        f"replaceWhere = {replace_predicate}"
    )


    (
        gold_df

        .write
        .format("delta")
        .mode("overwrite")

        # IMPORTANT:
        # No mergeSchema.
        # No overwriteSchema.
        # Existing schema is preserved.
        .option(
            "replaceWhere",
            replace_predicate
        )

        .saveAsTable(TARGET_TABLE)
    )


    log(
        "Atomic Gold replacement committed."
    )


# ============================================================
# POST-WRITE ROW VALIDATION
# ============================================================

post_write_df = (
    spark.table(TARGET_TABLE)
    .filter(
        F.col("source_date").cast("string")
        == TARGET_DATE
    )
)


post_count = post_write_df.count()


log(
    f"Post-write validation: "
    f"expected={gold_count}, "
    f"actual={post_count}"
)


if post_count != gold_count:

    fail(
        "Post-write count mismatch: "
        f"expected={gold_count}, "
        f"actual={post_count}"
    )


# ============================================================
# POST-WRITE SALES VALIDATION
# ============================================================

post_write_sales_row = (
    post_write_df

    .agg(
        F.sum("sales").alias("sales")
    )

    .first()
)


post_write_sales = float(
    post_write_sales_row["sales"] or 0.0
)


post_write_difference = abs(
    gold_sales - post_write_sales
)


log(
    f"Gold before write sales : "
    f"{gold_sales:.4f}"
)

log(
    f"Gold after write sales  : "
    f"{post_write_sales:.4f}"
)

log(
    f"Post-write difference   : "
    f"{post_write_difference:.4f}"
)


if post_write_difference > 0.01:

    fail(
        "Post-write sales validation failed. "
        f"Difference={post_write_difference:.4f}"
    )


log(
    "Post-write sales validation passed"
)


# ============================================================
# FINAL SCHEMA VALIDATION
# ============================================================

final_schema = spark.table(TARGET_TABLE).schema

final_schema_dict = {
    field.name: field.dataType.simpleString()
    for field in final_schema.fields
}


for column_name, expected_type in EXPECTED_TYPES.items():

    actual_type = final_schema_dict[column_name]

    if actual_type != expected_type:

        fail(
            f"Post-write schema mismatch for "
            f"{column_name}: "
            f"expected={expected_type}, "
            f"actual={actual_type}"
        )


log(
    "Post-write schema validation passed"
)


# ============================================================
# COMPLETE
# ============================================================

duration = (
    datetime.now(timezone.utc)
    - START_TIME
).total_seconds()


log("=" * 70)
log(
    "GOLD PRODUCT PERFORMANCE COMPLETED SUCCESSFULLY"
)
log(f"Pipeline run ID : {PIPELINE_RUN_ID}")
log(f"Target date     : {TARGET_DATE}")
log(f"Gold rows       : {gold_count}")
log(f"Total sales     : {gold_sales:.2f}")
log(f"Duration        : {duration:.2f}s")
log("=" * 70)

[GOLD_PRODUCT] ======================================================================
[GOLD_PRODUCT] PRODUCTION GOLD PRODUCT PERFORMANCE
[GOLD_PRODUCT] Pipeline run ID : 3efcad71-a0ff-44d8-8816-a42a520b3a4b
[GOLD_PRODUCT] Target date     : 2026-10-03
[GOLD_PRODUCT] Source table    : billing.sales_data.billing_silver
[GOLD_PRODUCT] Target table    : billing.sales_data.gold_product_performance
[GOLD_PRODUCT] ======================================================================
[GOLD_PRODUCT] Silver rows: 52995
[GOLD_PRODUCT] Rows after exclusions: 52995
[GOLD_PRODUCT] Generated Gold schema validation passed
[GOLD_PRODUCT] Gold rows: 8373
[GOLD_PRODUCT] Duplicate grain validation passed
[GOLD_PRODUCT] Rows belonging to excluded stores in Silver: 0
[GOLD_PRODUCT] Silver sales: 4496998.8700
[GOLD_PRODUCT] Gold sales  : 4496998.8700
[GOLD_PRODUCT] Difference  : 0.0000
[GOLD_PRODUCT] Silver -> Gold sales reconciliation passed
[GOLD_PRODUCT] Existing target columns: ['productName', 'orderDate

✅ Config loaded : billing.sales_data
📅 Target date   : 2026-10-03
